In [ ]:
# =============================================================
# NOTEBOOK RESUELTO — Assessment Técnico Galilei
# =============================================================
# Autor: Candidato
# Contexto: Datos del mercado eléctrico chileno — dataset de juguete
# con 6 registros por fuente, diseñado para evaluar criterio, no volumen.
# =============================================================

In [1]:
import pandas as pd
import sqlite3
import json
import numpy as np
from datetime import datetime

# ── Datos originales (sin modificar) ──────────────────────────
precios_raw = pd.DataFrame({
    'fecha': ['2024-01-01','2024-01-01','2024-01-02','2024-01-02','2024-01-03','2024-01-03'],
    'hora': [8, 20, 8, 20, 8, 20],
    'barra': ['Quillota','Alto Jahuel','Quillota','Alto Jahuel','Quillota','Alto Jahuel'],
    'precio_spot': [45.2, 51.8, None, 49.3, 60.1, None],
    'moneda': ['USD','USD','USD','USD','USD','USD']
})

generacion_raw = pd.DataFrame({
    'timestamp': ['2024-01-01 08:00','2024-01-01 20:00','2024-01-02 08:00',
                  '2024-01-02 20:00','2024-01-03 08:00','2024-01-03 20:00'],
    'central': ['Los Cóndores','Los Cóndores','Volcán','Volcán','Los Cóndores','Volcán'],
    'tipo': ['hidro','hidro','eólica','eólica','hidro','eólica'],
    'mwh_generados': [320, 280, 150, 170, 410, 'ERROR'],
    'barra_conexion': ['QUILLOTA-123','QUILLOTA-123','ALTO_JAHUEL-456',
                       'ALTO_JAHUEL-456','QUILLOTA-123','ALTO_JAHUEL-456']
})

tipo_cambio_json = '''
[
  {"fecha": "2024-01-01", "usd_clp": 895.0},
  {"fecha": "2024-01-02", "usd_clp": 901.5},
  {"fecha": "2024-01-03", "usd_clp": 899.0}
]
'''

In [2]:
# =============================================================
# PARTE 0: CALIDAD DE DATOS (bonus)
# =============================================================
# Decisión de diseño: separar validaciones en BLOQUEANTES vs ADVERTENCIAS.
# Bloqueante = no se puede operar con ese dato (ej: string donde debe haber número).
# Advertencia = dato faltante pero recuperable (ej: None en precio).
#
# Limitación conocida: con 6 filas, los umbrales estadísticos (z-score, IQR)
# no tienen poder. Por eso usamos validaciones por reglas de negocio.
# =============================================================

print("=" * 60)
print("VALIDACIÓN: precios_raw")
print("=" * 60)

# ── Check 1: Nulos ────────────────────────────────────────────
nulos_precio = precios_raw['precio_spot'].isna().sum()
print(f"[ADVERTENCIA] precio_spot tiene {nulos_precio} nulos de {len(precios_raw)} filas.")
print("  → No es bloqueante: dejaremos NaN explícito y en producción se interpolaría.")

# ── Check 2: Precios negativos (no tiene sentido en el mercado eléctrico)
# Nota: en mercados eléctricos el precio negativo SÍ puede ocurrir en exceso de oferta,
# pero en el SEN chileno es raro. Lo tratamos como advertencia.
negativos = (precios_raw['precio_spot'] < 0).sum()
if negativos > 0:
    print(f"[ADVERTENCIA] {negativos} precios negativos — revisar si son válidos.")
else:
    print("[OK] No hay precios negativos.")

# ── Check 3: Duplicados por llave natural (fecha+hora+barra)
dupes = precios_raw.duplicated(subset=['fecha','hora','barra']).sum()
print(f"[{'BLOQUEANTE' if dupes > 0 else 'OK'}] Duplicados en llave natural: {dupes}")

print()
print("=" * 60)
print("VALIDACIÓN: generacion_raw")
print("=" * 60)

# ── Check 4: String 'ERROR' en columna numérica ───────────────
# Esto es BLOQUEANTE: hace que toda la columna sea object, rompiendo cualquier cálculo.
errores = (generacion_raw['mwh_generados'] == 'ERROR').sum()
print(f"[BLOQUEANTE] mwh_generados contiene {errores} valor(es) 'ERROR' como string.")
print("  → Se convertirán a NaN con pd.to_numeric(..., errors='coerce').")

# ── Check 5: Tipo de dato de mwh_generados
print(f"[INFO] dtype actual de mwh_generados: {generacion_raw['mwh_generados'].dtype}")
print("  → Debería ser float64, no object.")

# ── Check 6: Inconsistencia en nombres de barra
barras_gen = set(generacion_raw['barra_conexion'].unique())
barras_pre = set(precios_raw['barra'].unique())
print(f"[BLOQUEANTE] barras en generacion: {barras_gen}")
print(f"             barras en precios:    {barras_pre}")
print("  → No coinciden. El merge daría 0 filas sin normalizar.")

print()
print("=" * 60)
print("VALIDACIÓN: tipo_cambio_json")
print("=" * 60)

# ── Check 7: Parseo del JSON
try:
    tc_parsed = json.loads(tipo_cambio_json)
    print(f"[OK] JSON válido. {len(tc_parsed)} registros.")
except json.JSONDecodeError as e:
    print(f"[BLOQUEANTE] JSON inválido: {e}")

# ── Check 8: Cobertura de fechas entre fuentes
fechas_tc = {r['fecha'] for r in tc_parsed}
fechas_pre = set(precios_raw['fecha'].unique())
if fechas_pre == fechas_tc:
    print("[OK] Cobertura de fechas completa entre tipo de cambio y precios.")
else:
    print(f"[ADVERTENCIA] Fechas sin tipo de cambio: {fechas_pre - fechas_tc}")

VALIDACIÓN: precios_raw
[ADVERTENCIA] precio_spot tiene 2 nulos de 6 filas.
  → No es bloqueante: dejaremos NaN explícito y en producción se interpolaría.
[OK] No hay precios negativos.
[OK] Duplicados en llave natural: 0

VALIDACIÓN: generacion_raw
[BLOQUEANTE] mwh_generados contiene 1 valor(es) 'ERROR' como string.
  → Se convertirán a NaN con pd.to_numeric(..., errors='coerce').
[INFO] dtype actual de mwh_generados: object
  → Debería ser float64, no object.
[BLOQUEANTE] barras en generacion: {'ALTO_JAHUEL-456', 'QUILLOTA-123'}
             barras en precios:    {'Alto Jahuel', 'Quillota'}
  → No coinciden. El merge daría 0 filas sin normalizar.

VALIDACIÓN: tipo_cambio_json
[OK] JSON válido. 3 registros.
[OK] Cobertura de fechas completa entre tipo de cambio y precios.


In [3]:
# =============================================================
# PARTE 1: LIMPIEZA DE DATOS
# =============================================================
# Principio: no modificar los raw. Crear copias limpias.
# Cada decisión está documentada con su justificación.
# =============================================================

# ── Limpiar precios_raw ───────────────────────────────────────
precios = precios_raw.copy()

# Convertir fecha a datetime — necesario para join temporal y operaciones de fecha
precios['fecha'] = pd.to_datetime(precios['fecha'])

# Dejar los nulos en precio_spot como NaN explícito.
# DECISIÓN: No interpolamos con 6 filas — una interpolación con tan pocos
# puntos introduce más ruido que información. En producción usaríamos
# forward-fill por barra o promedio histórico de la misma hora.
# Los NaN son visibles y no silenciosos, lo que facilita el debugging.

print("precios limpio:")
print(precios.dtypes)
print(precios)

print()

# ── Limpiar generacion_raw ────────────────────────────────────
generacion = generacion_raw.copy()

# PROBLEMA CLAVE: 'ERROR' como string hace que toda la columna sea object.
# errors='coerce' convierte valores no numéricos a NaN — seguro y explícito.
generacion['mwh_generados'] = pd.to_numeric(generacion['mwh_generados'], errors='coerce')

# Parsear timestamp a datetime y extraer fecha y hora por separado
# para poder hacer el join con precios (que tiene fecha+hora como columnas separadas)
generacion['timestamp'] = pd.to_datetime(generacion['timestamp'])
generacion['fecha'] = generacion['timestamp'].dt.normalize()  # date sin hora
generacion['hora'] = generacion['timestamp'].dt.hour

# NORMALIZAR barra_conexion → formato igual a precios['barra']
# 'QUILLOTA-123' → quitar sufijo (-\d+), quitar guiones bajos, title case
# Usamos regex para quitar el patrón '-XXX' al final
generacion['barra'] = (
    generacion['barra_conexion']
    .str.replace(r'-\d+$', '', regex=True)   # quita '-123', '-456'
    .str.replace('_', ' ')                    # ALTO_JAHUEL → ALTO JAHUEL
    .str.title()                              # ALTO JAHUEL → Alto Jahuel
)

print("generacion limpia:")
print(generacion[['fecha','hora','central','tipo','mwh_generados','barra']])
print("\nbarras normalizadas:", generacion['barra'].unique())

print()

# ── Limpiar tipo_cambio_json ──────────────────────────────────
# Parsear el string JSON a DataFrame
tipo_cambio = pd.DataFrame(json.loads(tipo_cambio_json))
tipo_cambio['fecha'] = pd.to_datetime(tipo_cambio['fecha'])

print("tipo_cambio limpio:")
print(tipo_cambio)

precios limpio:
fecha          datetime64[ns]
hora                    int64
barra                  object
precio_spot           float64
moneda                 object
dtype: object
       fecha  hora        barra  precio_spot moneda
0 2024-01-01     8     Quillota         45.2    USD
1 2024-01-01    20  Alto Jahuel         51.8    USD
2 2024-01-02     8     Quillota          NaN    USD
3 2024-01-02    20  Alto Jahuel         49.3    USD
4 2024-01-03     8     Quillota         60.1    USD
5 2024-01-03    20  Alto Jahuel          NaN    USD

generacion limpia:
       fecha  hora       central    tipo  mwh_generados        barra
0 2024-01-01     8  Los Cóndores   hidro          320.0     Quillota
1 2024-01-01    20  Los Cóndores   hidro          280.0     Quillota
2 2024-01-02     8        Volcán  eólica          150.0  Alto Jahuel
3 2024-01-02    20        Volcán  eólica          170.0  Alto Jahuel
4 2024-01-03     8  Los Cóndores   hidro          410.0     Quillota
5 2024-01-03    20    

In [4]:
# =============================================================
# PARTE 2: INTEGRACIÓN
# =============================================================
# Construimos el DataFrame `mercado` cruzando las tres fuentes.
#
# Decisiones de diseño:
#   - JOIN generacion ← precios: usamos LEFT JOIN para no perder
#     filas de generación aunque no haya precio (precio queda NaN).
#     Así el dato de generación siempre es visible aunque falte precio.
#   - JOIN resultado ← tipo_cambio: INNER JOIN porque sin tipo de cambio
#     no podemos calcular precio en CLP, pero en este dataset la cobertura
#     es completa así que no perdemos filas.
#   - Columna ingreso_estimado_clp: MWh × precio_USD × tipo_cambio.
#     Es una aproximación — en el mercado real habría que considerar
#     contratos, peajes y otros factores.
# =============================================================

# Paso 1: unir generación con precios por fecha + hora + barra
mercado = generacion.merge(
    precios[['fecha','hora','barra','precio_spot','moneda']],
    on=['fecha','hora','barra'],
    how='left'  # no perder filas de generación sin precio
)

# Paso 2: agregar tipo de cambio por fecha
mercado = mercado.merge(
    tipo_cambio,
    on='fecha',
    how='left'
)

# Paso 3: calcular ingreso estimado en CLP
# Solo se puede calcular donde precio_spot y mwh_generados no son NaN
mercado['ingreso_clp'] = mercado['mwh_generados'] * mercado['precio_spot'] * mercado['usd_clp']

# Paso 4: columna de periodo (útil para análisis peak/off-peak)
# En el mercado eléctrico chileno, hora 8 = bloque diurno, hora 20 = nocturno
mercado['periodo'] = mercado['hora'].map({8: 'peak', 20: 'off-peak'})

print("DataFrame mercado integrado:")
print(mercado[['fecha','hora','periodo','central','tipo','barra',
               'mwh_generados','precio_spot','usd_clp','ingreso_clp']].to_string())

print(f"\nFilas con precio_spot nulo: {mercado['precio_spot'].isna().sum()}")
print(f"Filas con mwh nulo: {mercado['mwh_generados'].isna().sum()}")
print(f"Filas con ingreso calculable: {mercado['ingreso_clp'].notna().sum()} de {len(mercado)}")

DataFrame mercado integrado:
       fecha  hora   periodo       central    tipo        barra  mwh_generados  precio_spot  usd_clp  ingreso_clp
0 2024-01-01     8      peak  Los Cóndores   hidro     Quillota          320.0         45.2    895.0   12945280.0
1 2024-01-01    20  off-peak  Los Cóndores   hidro     Quillota          280.0          NaN    895.0          NaN
2 2024-01-02     8      peak        Volcán  eólica  Alto Jahuel          150.0          NaN    901.5          NaN
3 2024-01-02    20  off-peak        Volcán  eólica  Alto Jahuel          170.0         49.3    901.5    7555471.5
4 2024-01-03     8      peak  Los Cóndores   hidro     Quillota          410.0         60.1    899.0   22152259.0
5 2024-01-03    20  off-peak        Volcán  eólica  Alto Jahuel            NaN          NaN    899.0          NaN

Filas con precio_spot nulo: 3
Filas con mwh nulo: 1
Filas con ingreso calculable: 3 de 6


In [5]:
# =============================================================
# PARTE 3: SQL
# =============================================================
# Persistimos mercado en SQLite y ejecutamos queries analíticas.
#
# Por qué estas métricas:
#   - Precio promedio por barra: métrica de referencia para contratos
#   - Generación por tipo: base para análisis de despacho
#   - Precio peak vs off-peak: diferencial de valor entre bloques horarios
#   - Ingreso por central: para rentabilidad estimada
# =============================================================

# Persistir en SQLite (en memoria para este notebook)
conn = sqlite3.connect(':memory:')

# Convertir fecha a string para SQLite (no maneja datetime nativo)
mercado_sql = mercado.copy()
mercado_sql['fecha'] = mercado_sql['fecha'].astype(str)
mercado_sql.to_sql('mercado', conn, index=False, if_exists='replace')

print("Tabla 'mercado' creada en SQLite.\n")

# ── Query 1: Precio promedio por barra ──────────────────────
q1 = """
SELECT
    barra,
    ROUND(AVG(precio_spot), 2) AS precio_promedio_usd,
    COUNT(*) AS n_observaciones,
    COUNT(precio_spot) AS n_con_precio  -- filas donde precio no es NULL
FROM mercado
GROUP BY barra
ORDER BY precio_promedio_usd DESC
"""
print("Q1 — Precio promedio por barra:")
print(pd.read_sql(q1, conn))

# ── Query 2: Generación total por tipo de central ────────────
q2 = """
SELECT
    tipo,
    barra,
    ROUND(SUM(mwh_generados), 1) AS mwh_total,
    COUNT(*) AS bloques
FROM mercado
WHERE mwh_generados IS NOT NULL
GROUP BY tipo, barra
ORDER BY mwh_total DESC
"""
print("\nQ2 — Generación por tipo y barra:")
print(pd.read_sql(q2, conn))

# ── Query 3: Diferencial peak vs off-peak por barra ──────────
# Relevante para el mercado chileno: los contratos PPA típicamente
# distinguen entre bloques horarios con precios distintos.
q3 = """
SELECT
    barra,
    periodo,
    ROUND(AVG(precio_spot), 2) AS precio_promedio,
    ROUND(SUM(mwh_generados), 1) AS mwh_total
FROM mercado
GROUP BY barra, periodo
ORDER BY barra, periodo
"""
print("\nQ3 — Peak vs Off-peak por barra:")
print(pd.read_sql(q3, conn))

# ── Query 4: Ingreso estimado por central (top line) ─────────
q4 = """
SELECT
    central,
    tipo,
    ROUND(SUM(mwh_generados), 1) AS mwh_total,
    ROUND(SUM(ingreso_clp) / 1e6, 2) AS ingreso_mm_clp,  -- en millones de CLP
    COUNT(*) AS bloques_operados
FROM mercado
WHERE ingreso_clp IS NOT NULL
GROUP BY central, tipo
ORDER BY ingreso_mm_clp DESC
"""
print("\nQ4 — Ingreso estimado por central (MM CLP):")
print(pd.read_sql(q4, conn))

conn.close()

Tabla 'mercado' creada en SQLite.

Q1 — Precio promedio por barra:
         barra  precio_promedio_usd  n_observaciones  n_con_precio
0     Quillota                52.65                3             2
1  Alto Jahuel                49.30                3             1

Q2 — Generación por tipo y barra:
     tipo        barra  mwh_total  bloques
0   hidro     Quillota     1010.0        3
1  eólica  Alto Jahuel      320.0        2

Q3 — Peak vs Off-peak por barra:
         barra   periodo  precio_promedio  mwh_total
0  Alto Jahuel  off-peak            49.30      170.0
1  Alto Jahuel      peak              NaN      150.0
2     Quillota  off-peak              NaN      280.0
3     Quillota      peak            52.65      730.0

Q4 — Ingreso estimado por central (MM CLP):
        central    tipo  mwh_total  ingreso_mm_clp  bloques_operados
0  Los Cóndores   hidro      730.0           35.10                 2
1        Volcán  eólica      170.0            7.56                 1


In [6]:
# =============================================================
# PARTE 4: DETECCIÓN DE ANOMALÍAS
# =============================================================
# Con 6 filas NO usamos métodos estadísticos (z-score, IQR):
# son inestables con tan pocos datos. Usamos reglas de negocio
# que serían válidas independientemente del tamaño del dataset.
#
# CRITERIOS:
#   1. Datos inválidos estructurales (tipos incorrectos) → BLOQUEANTE
#   2. Nulos en campos críticos → ADVERTENCIA
#   3. Valores fuera de rango físico/comercial → ADVERTENCIA
#   4. Saltos bruscos entre timestamps consecutivos → ADVERTENCIA
#
# LIMITACIONES:
#   - Los umbrales de rango (ej: precio > 200 USD/MWh) son heurísticos
#     basados en rangos típicos del SEN. En producción deben calibrarse.
#   - Los saltos bruscos con 6 filas solo detectan 1 cambio: poco poder.
#   - No detectamos seasonalidad ni patrones — necesitaríamos meses de datos.
# =============================================================

anomalias = []  # lista de dicts con {tipo, severidad, detalle, fila}

def registrar(tipo, severidad, detalle, fila=None):
    """Registra una anomalía con su metadata."""
    anomalias.append({
        'tipo': tipo,
        'severidad': severidad,  # 'BLOQUEANTE' o 'ADVERTENCIA'
        'detalle': detalle,
        'fila': fila
    })

# ── Regla 1: Nulos en precio_spot ────────────────────────────
for idx, row in precios.iterrows():
    if pd.isna(row['precio_spot']):
        registrar(
            'precio_nulo', 'ADVERTENCIA',
            f"precio_spot nulo en {row['fecha'].date()} hora {row['hora']} barra {row['barra']}",
            fila=idx
        )

# ── Regla 2: Precio fuera de rango SEN ───────────────────────
# En el SEN, precios típicos están entre 10 y 250 USD/MWh.
# Fuera de ese rango es posible pero merece revisión.
PRECIO_MIN, PRECIO_MAX = 10, 250
for idx, row in precios.iterrows():
    if pd.notna(row['precio_spot']):
        if not (PRECIO_MIN <= row['precio_spot'] <= PRECIO_MAX):
            registrar(
                'precio_fuera_rango', 'ADVERTENCIA',
                f"precio {row['precio_spot']} USD fuera de [{PRECIO_MIN},{PRECIO_MAX}] "
                f"en {row['fecha'].date()} hora {row['hora']}",
                fila=idx
            )

# ── Regla 3: MWh nulo o negativo ─────────────────────────────
for idx, row in generacion.iterrows():
    if pd.isna(row['mwh_generados']):
        registrar(
            'mwh_nulo', 'ADVERTENCIA',
            f"mwh_generados nulo en {row['central']} {row['timestamp']}",
            fila=idx
        )
    elif row['mwh_generados'] < 0:
        registrar(
            'mwh_negativo', 'BLOQUEANTE',
            f"mwh_generados negativo ({row['mwh_generados']}) en {row['central']}",
            fila=idx
        )

# ── Regla 4: Saltos bruscos entre timestamps para la misma central ───
# Un cambio > 50% entre bloques consecutivos puede indicar error o evento extremo
UMBRAL_SALTO = 0.50  # 50%
for central, grupo in generacion.groupby('central'):
    grupo = grupo.sort_values('timestamp')
    mwh = grupo['mwh_generados'].dropna()
    if len(mwh) >= 2:
        cambio_pct = mwh.pct_change().abs()
        for idx, pct in cambio_pct.items():
            if pd.notna(pct) and pct > UMBRAL_SALTO:
                registrar(
                    'salto_generacion', 'ADVERTENCIA',
                    f"Central {central}: cambio de {pct*100:.0f}% entre bloques consecutivos",
                    fila=idx
                )

# ── Reporte final ─────────────────────────────────────────────
df_anomalias = pd.DataFrame(anomalias)
print("=" * 60)
print("REPORTE DE ANOMALÍAS")
print("=" * 60)
if df_anomalias.empty:
    print("No se detectaron anomalías.")
else:
    print(df_anomalias.to_string(index=False))
    print(f"\nResumen: {(df_anomalias['severidad']=='BLOQUEANTE').sum()} bloqueantes, "
          f"{(df_anomalias['severidad']=='ADVERTENCIA').sum()} advertencias")

REPORTE DE ANOMALÍAS
       tipo   severidad                                                  detalle  fila
precio_nulo ADVERTENCIA     precio_spot nulo en 2024-01-02 hora 8 barra Quillota     2
precio_nulo ADVERTENCIA precio_spot nulo en 2024-01-03 hora 20 barra Alto Jahuel     5
   mwh_nulo ADVERTENCIA         mwh_generados nulo en Volcán 2024-01-03 20:00:00     5

Resumen: 0 bloqueantes, 3 advertencias
